In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Represents the demand side - customers seeking insurance quotes online.

## Business Use Cases </br>
**Lead qualification**: Understanding lead value and complexity

**Segmentation:** Grouping leads by characteristics

**Behavioral analysis:** Digital engagement patterns

**Seasonal patterns:** Understanding demand fluctuations

In [2]:
df = pd.read_csv("synthetic_leads_v80.csv")
df.head()

,lead_id,lead_date,region,postal_code_prefix,insurance_type,language,tenure_years,digital_engagement_score,quote_value,lead_difficulty,sophistication,patience_hours,claims_severity,multi_product_intent,hour_of_day,is_weekend,month
0,LD-000003,2023-05-10,Mississauga,MIS,home,English,1.0,37.8,790.65,0.752,0.202,37.8,none,False,10,False,5
1,LD-000005,2024-10-18,Toronto,TOR,bundle,English,3.0,25.8,2543.51,0.844,0.444,15.9,none,True,10,False,10
2,LD-000008,2024-05-10,Mississauga,MIS,home,English,0.0,34.2,594.89,0.888,0.499,11.7,minor,False,18,False,5
3,LD-000009,2025-05-07,Other_GTA,OTH,auto,English,1.0,29.7,1465.74,1.209,0.363,75.8,major,False,19,False,5
4,LD-000013,2025-06-30,Toronto,TOR,bundle,English,0.0,59.3,2168.47,1.097,0.219,6.0,none,True,9,False,6


In [3]:
df.shape

(10624, 17)

In [5]:
df.columns

Index(['lead_id', 'lead_date', 'region', 'postal_code_prefix',
       'insurance_type', 'language', 'tenure_years',
       'digital_engagement_score', 'quote_value', 'lead_difficulty',
       'sophistication', 'patience_hours', 'claims_severity',
       'multi_product_intent', 'hour_of_day', 'is_weekend', 'month'],
      dtype='str')

lead_id                 # LD-000001, LD-000002, etc. </br>

lead_date               # Timestamp of lead generation

region                  # Geographic location

postal_code_prefix      # First 3 characters of postal code

insurance_type          # auto, home, or bundle

language                # English or French

tenure_years            # Years as existing customer (0-5)

digital_engagement_score # Digital interaction intensity (0-100)

quote_value             # Expected premium amount ($)

lead_difficulty         # How complex the lead is (0.7-1.3)

sophistication          # Customer's insurance knowledge (0-1)

patience_hours          # How long willing to wait for response (6-168)

claims_severity         # none, minor, or major claims history

multi_product_intent    # Boolean - wants to bundle multiple products

hour_of_day             # When lead was generated (9-20)

is_weekend              # Boolean

month                   # Seasonality factor

In [ ]:
import pandas as pd

# ==============================
# 1. LOAD DATA
# ==============================
brokers = pd.read_csv("synthetic_brokers_v80.csv")
leads = pd.read_csv("synthetic_leads_v80.csv")
assignments = pd.read_csv("synthetic_assignments_v80.csv")
counterfactual = pd.read_csv("synthetic_counterfactual_v80.csv")
historical = pd.read_csv("synthetic_historical_v80.csv")

datasets = {
    "brokers": brokers,
    "leads": leads,
    "assignments": assignments,
    "counterfactual": counterfactual,
    "historical": historical
}

# ==============================
# 2. SHAPE OVERVIEW
# ==============================
print("\n=== DATASET SHAPES ===")
for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

# ==============================
# 3. DATA TYPES
# ==============================
print("\n=== DATA TYPES ===")
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.dtypes)

# ==============================
# 4. PRIMARY KEY CHECK
# ==============================
print("\n=== PRIMARY KEY UNIQUENESS ===")

print("brokers:", brokers['broker_id'].is_unique)
print("leads:", leads['lead_id'].is_unique)

print("assignments (lead_id + broker_id + interaction):",
      assignments[['lead_id', 'broker_id', 'interaction_number']].duplicated().sum() == 0)

print("counterfactual (lead_id + broker_id):",
      counterfactual[['lead_id', 'broker_id']].duplicated().sum() == 0)

# ==============================
# 5. DUPLICATE CHECK
# ==============================
print("\n=== DUPLICATE ROWS ===")
for name, df in datasets.items():
    dup_count = df.duplicated().sum()
    print(f"{name}: {dup_count} duplicate rows")

# ==============================
# 6. FOREIGN KEY RELATIONSHIPS
# ==============================
print("\n=== FOREIGN KEY VALIDATION ===")

# assignments -> leads
missing_leads = assignments[~assignments['lead_id'].isin(leads['lead_id'])].shape[0]
print(f"Assignments with missing lead_id: {missing_leads}")

# assignments -> brokers
missing_brokers = assignments[~assignments['broker_id'].isin(brokers['broker_id'])].shape[0]
print(f"Assignments with missing broker_id: {missing_brokers}")

# counterfactual -> leads
missing_cf_leads = counterfactual[~counterfactual['lead_id'].isin(leads['lead_id'])].shape[0]
print(f"Counterfactual missing lead_id: {missing_cf_leads}")

# ==============================
# 7. TEMPORAL ANALYSIS
# ==============================
print("\n=== TEMPORAL RANGE ===")

# Convert to datetime
leads['lead_date'] = pd.to_datetime(leads['lead_date'])
assignments['assignment_date'] = pd.to_datetime(assignments['assignment_date'])

print(f"Lead date range: {leads['lead_date'].min()} → {leads['lead_date'].max()}")
print(f"Assignment date range: {assignments['assignment_date'].min()} → {assignments['assignment_date'].max()}")

# Check logical consistency
invalid_dates = assignments[assignments['assignment_date'] < assignments['assignment_date'].min()]
print(f"Invalid assignment dates: {len(invalid_dates)}")

# ==============================
# 8. BASIC TABLE PREVIEW
# ==============================
print("\n=== SAMPLE RECORDS ===")
for name, df in datasets.items():
    print(f"\n{name.upper()} SAMPLE:")
    print(df.head(3))

In [6]:
data = pd.read_csv("synthetic_assignments_v80.csv")
data.head()

,lead_id,original_lead_id,broker_id,assignment_date,converted,censored,conversion_delay_days,conversion_value,revenue,cost,...,expected_profit,is_assigned,interaction_number,responded,response_time_hours,propensity_score,exploration_rate_at_time,action_probabilities,position_bias,market_regime
0,LD-000003,LD-000003,BR-0218,2023-05-10 00:00:00,0,0,NaN,0.0,0.0,62.97,...,-62.97,1,0,0.0,NaN,0.098307,0.1485,"{""BR-0101"": 0.10559891012488674, ""BR-0039"": 0....",0.0000,normal
1,LD-000003,LD-000003,BR-0138,2023-05-10 00:00:00,0,0,NaN,0.0,0.0,80.48,...,-70.63,1,1,1.0,5.9,0.100927,0.1485,"{""BR-0101"": 0.10615076555011208, ""BR-0039"": 0....",0.7408,normal
2,LD-000003,LD-000003,BR-0039,2023-05-10 00:00:00,0,0,NaN,0.0,0.0,0.00,...,10.86,0,-1,NaN,NaN,0.000000,0.1485,{},0.0000,normal
3,LD-000003,LD-000003,BR-0078,2023-05-10 02:00:00,0,0,NaN,0.0,0.0,0.00,...,8.54,0,-1,NaN,NaN,0.000000,0.1485,{},0.0000,normal
4,LD-000003,LD-000003,BR-0283,2023-05-10 00:00:00,0,0,NaN,0.0,0.0,0.00,...,7.62,0,-1,NaN,NaN,0.000000,0.1485,{},0.0000,normal


This is your historical log of all matching decisions and outcomes. It's the primary training data for  recommendation system.

In [7]:
data.columns

Index(['lead_id', 'original_lead_id', 'broker_id', 'assignment_date',
       'converted', 'censored', 'conversion_delay_days', 'conversion_value',
       'revenue', 'cost', 'profit', 'expected_profit', 'is_assigned',
       'interaction_number', 'responded', 'response_time_hours',
       'propensity_score', 'exploration_rate_at_time', 'action_probabilities',
       'position_bias', 'market_regime'],
      dtype='str')

# Identification </br>
lead_id                 # Which lead was assigned </br>
broker_id               # Which broker received the lead </br>
assignment_date         # When assignment happened </br>
original_lead_id        # For re-entry tracking </br>

# Assignment Context </br>
is_assigned             # 1 for actual assignments, 0 for counterfactual </br>
interaction_number      # Which attempt in the journey (0,1,2) </br>
responded               # Did broker respond? (0/1) </br>
response_time_hours     # Hours until response </br>

# Outcome </br>
converted               # Did lead convert to policy? (0/1) </br>
censored                # 1 if conversion happened after observation window </br>
conversion_delay_days   # Days until conversion </br>
conversion_value        # Premium amount if converted </br>

# Financials </br>
revenue                 # Commission earned if converted </br>
cost                    # Cost per lead </br>
profit                  # Revenue - cost </br>
expected_profit         # Probability-weighted expected profit </br>

# Causal Inference (CRITICAL for unbiased learning)</br>
propensity_score        # Probability this assignment was chosen (0.01-0.99) </br>
action_probabilities    # JSON of all broker probabilities at decision time </br>
exploration_rate_at_time # Current exploration rate (0.05-0.15) </br>
position_bias           # Position effect in ranking (0.3-1.0)</br>
market_regime           # normal, hard, or soft market conditions </br>

In [9]:
data.shape

(54136, 21)

In [10]:
data.describe()

,converted,censored,conversion_delay_days,conversion_value,revenue,cost,profit,expected_profit,is_assigned,interaction_number,responded,response_time_hours,propensity_score,exploration_rate_at_time,position_bias
count,54136.000000,54136.0,2377.000000,54136.000000,54136.000000,54136.000000,54136.000000,54136.000000,54136.000000,54136.000000,20354.000000,16873.000000,54136.000000,54136.000000,54136.000000
mean,0.043908,0.0,2.868616,65.884603,7.514957,37.352955,-29.837998,-14.987841,0.375979,-0.397647,0.828977,7.068014,0.039377,0.050394,0.266350
std,0.204892,0.0,2.942935,330.070790,38.475203,51.317814,56.293765,54.518102,0.484379,0.899625,0.376538,8.674392,0.051902,0.004847,0.407472
min,0.000000,0.0,0.100000,0.000000,0.000000,0.000000,-149.170000,-149.170000,0.000000,-1.000000,0.000000,0.500000,0.000000,0.050000,0.000000
25%,0.000000,0.0,0.800000,0.000000,0.000000,0.000000,-72.360000,-62.580000,0.000000,-1.000000,1.000000,1.600000,0.000000,0.050000,0.000000
50%,0.000000,0.0,1.900000,0.000000,0.000000,0.000000,0.000000,12.830000,0.000000,-1.000000,1.000000,4.100000,0.000000,0.050000,0.000000
75%,0.000000,0.0,4.000000,0.000000,0.000000,80.480000,0.000000,23.390000,1.000000,0.000000,1.000000,9.100000,0.098734,0.050000,0.740800
max,1.000000,0.0,25.000000,2993.420000,442.993600,149.170000,382.803600,145.910000,1.000000,2.000000,1.000000,72.000000,0.303779,0.148500,1.000000


In [11]:
counterfactual_data  = pd.read_csv('synthetic_counterfactual_v80.csv')
counterfactual_data.head()

,lead_id,original_lead_id,broker_id,potential_outcome,potential_conversion_probability,potential_revenue,potential_profit,match_quality
0,LD-000003,LD-000003,BR-0039,0,0.1205,0.0,0.0,0.882398
1,LD-000003,LD-000003,BR-0078,0,0.0982,0.0,0.0,0.711194
2,LD-000003,LD-000003,BR-0283,0,0.0876,0.0,0.0,0.619689
3,LD-000005,LD-000005,BR-0031,0,0.0979,0.0,0.0,0.782767
4,LD-000005,LD-000005,BR-0135,0,0.1024,0.0,0.0,0.823027


Ground truth for model evaluation - shows what would have happened with alternative assignments.

lead_id                              # Which lead </br>
original_lead_id                     # Original lead (if re-entered)  </br>
broker_id                            # Counterfactual broker  </br>
potential_outcome                    # 1 if would have converted  </br>
potential_conversion_probability     # Probability of conversion  </br>
potential_revenue                    # Expected revenue  </br>
potential_profit                     # Expected profit  </br>
match_quality                        # True match quality (0-1)  </br>

In [12]:
synthetic_historical = pd.read_csv("synthetic_historical_v80.csv")
synthetic_historical.head()

/tmp/ipykernel_39490/1538316045.py:1: DtypeWarning: Columns (0: multi_product_intent, 1: is_weekend) have mixed types. Specify dtype option on import or set low_memory=False.
  synthetic_historical = pd.read_csv("synthetic_historical_v80.csv")


,lead_id,original_lead_id,broker_id,assignment_date,converted,censored,conversion_delay_days,conversion_value,revenue,cost,...,ribo_license_years,capacity,avg_response_time,is_new_broker,skill_level,reliability,commission_rate,cost_per_lead,efficiency,burnout_risk
0,LD-000003,LD-000003,BR-0218,2023-05-10 00:00:00,0,0,NaN,0.0,0.0,62.97,...,23.0,75.0,3.7,False,0.696,0.804,0.096,62.97,0.756,0.090
1,LD-000003,LD-000003,BR-0138,2023-05-10 00:00:00,0,0,NaN,0.0,0.0,80.48,...,13.0,40.0,3.2,False,0.900,0.882,0.131,80.48,1.329,0.289
2,LD-000003,LD-000003,BR-0039,2023-05-10 00:00:00,0,0,NaN,0.0,0.0,0.00,...,3.0,40.0,3.3,False,0.854,0.926,0.114,111.83,0.869,0.111
3,LD-000003,LD-000003,BR-0078,2023-05-10 02:00:00,0,0,NaN,0.0,0.0,0.00,...,19.0,60.0,5.1,False,0.889,0.980,0.110,70.05,0.992,0.206
4,LD-000003,LD-000003,BR-0283,2023-05-10 00:00:00,0,0,NaN,0.0,0.0,0.00,...,18.0,50.0,9.5,False,0.847,0.760,0.110,113.20,1.226,0.290


Merged training dataset - combines all other datasets for convenience. This is typically use for model training.

In [13]:
synthetic_historical.columns

Index(['lead_id', 'original_lead_id', 'broker_id', 'assignment_date',
       'converted', 'censored', 'conversion_delay_days', 'conversion_value',
       'revenue', 'cost', 'profit', 'expected_profit', 'is_assigned',
       'interaction_number', 'responded', 'response_time_hours',
       'propensity_score', 'exploration_rate_at_time', 'action_probabilities',
       'position_bias', 'market_regime', 'lead_date', 'region_x',
       'postal_code_prefix', 'insurance_type', 'language', 'tenure_years',
       'digital_engagement_score', 'quote_value', 'lead_difficulty',
       'sophistication', 'patience_hours', 'claims_severity',
       'multi_product_intent', 'hour_of_day', 'is_weekend', 'month',
       'region_y', 'expertise_auto', 'expertise_home', 'expertise_bundle',
       'conversion_rate', 'csat_score', 'languages', 'ribo_licensed',
       'ribo_license_years', 'capacity', 'avg_response_time', 'is_new_broker',
       'skill_level', 'reliability', 'commission_rate', 'cost_per_lead',
 

In [15]:
synthetic_historical.shape

(54136, 55)

In [14]:
synthetic_historical.describe()

,converted,censored,conversion_delay_days,conversion_value,revenue,cost,profit,expected_profit,is_assigned,interaction_number,...,csat_score,ribo_license_years,capacity,avg_response_time,skill_level,reliability,commission_rate,cost_per_lead,efficiency,burnout_risk
count,54136.000000,54136.0,2377.000000,54136.000000,54136.000000,54136.000000,54136.000000,54136.000000,54136.000000,54136.000000,...,53558.000000,53558.000000,53558.000000,53558.000000,53558.000000,53558.000000,53558.000000,53558.000000,53558.000000,53558.000000
mean,0.043908,0.0,2.868616,65.884603,7.514957,37.352955,-29.837998,-14.987841,0.375979,-0.397647,...,4.433877,10.864819,50.991262,6.605529,0.802502,0.846956,0.114563,98.329608,1.020512,0.148751
std,0.204892,0.0,2.942935,330.070790,38.475203,51.317814,56.293765,54.518102,0.484379,0.899625,...,0.446854,7.277720,11.249681,2.843670,0.119517,0.087914,0.020192,28.576430,0.281810,0.087130
min,0.000000,0.0,0.100000,0.000000,0.000000,0.000000,-149.170000,-149.170000,0.000000,-1.000000,...,2.980000,0.000000,40.000000,2.000000,0.360000,0.580000,0.080000,50.000000,0.500000,0.000000
25%,0.000000,0.0,0.800000,0.000000,0.000000,0.000000,-72.360000,-62.580000,0.000000,-1.000000,...,4.130000,5.000000,40.000000,4.000000,0.742000,0.794000,0.096000,72.360000,0.797000,0.070000
50%,0.000000,0.0,1.900000,0.000000,0.000000,0.000000,0.000000,12.830000,0.000000,-1.000000,...,4.480000,10.000000,50.000000,6.600000,0.842000,0.847000,0.115000,96.390000,1.019000,0.153000
75%,0.000000,0.0,4.000000,0.000000,0.000000,80.480000,0.000000,23.390000,1.000000,0.000000,...,4.820000,17.000000,60.000000,8.800000,0.900000,0.909000,0.132000,124.340000,1.262000,0.224000
max,1.000000,0.0,25.000000,2993.420000,442.993600,149.170000,382.803600,145.910000,1.000000,2.000000,...,5.000000,24.000000,75.000000,12.000000,0.900000,0.980000,0.150000,149.170000,1.495000,0.297000


In [ ]:

print(synthetic_historical.isnull().sum())


lead_id                         0
original_lead_id                0
broker_id                       0
assignment_date                 0
converted                       0
censored                        0
conversion_delay_days       51759
conversion_value                0
revenue                         0
cost                            0
profit                          0
expected_profit                 0
is_assigned                     0
interaction_number              0
responded                   33782
response_time_hours         37263
propensity_score                0
exploration_rate_at_time        0
action_probabilities            0
position_bias                   0
market_regime                   0
lead_date                    3993
region_x                     3993
postal_code_prefix           3993
insurance_type               5102
language                     5071
tenure_years                 4648
digital_engagement_score     4296
quote_value                  3993
lead_difficult